# K05 — Held-out corpora RAID and Beemo: outcomes and detector scores (stage S2/S5)

Builds the feature tables of two held-out corpora under the frozen protocol:

* **RAID** (training split): 5 domains, human documents, 4 generators, 3 adversarial attacks
  on 2 generators — about 37,500 documents (`d2vi.data.heldout`);
* **Beemo**: human answers, raw LLM outputs, expert-edited and LLM-edited outputs — about
  10,800 documents.

For every document: four outcomes (CPU), RoBERTa and RADAR scores, Fast-DetectGPT score
(GPU, cached every 500 documents). Thresholds and the TF-IDF detector follow the protocol
(cross-domain for RAID, 30% calibration split by prompt for Beemo).

**Settings:** Accelerator = GPU T4 x2, Internet = On. No input dataset is needed the first
time. **Run:** first run cells 1-3 by hand (about 3 minutes; cell 3 only checks the RAID file
format on a small download) — if they pass, stop the session and use *Save Version → Save &
Run All (Commit)*. Expected: 5-7 hours of GPU.

**Output to send back** (no text inside): `raid_features.parquet`, `beemo_features.parquet`,
`raid_summary.json`, `beemo_summary.json`.

**If the run stops:** create a private dataset from the files `k05_*` of the Output, add it
as Input, and run again; sampling and finished scores are not repeated.


In [ ]:
!pip -q install -U transformers accelerate vaderSentiment textstat nrclex
!pip -q install --force-reinstall --no-deps "d2vi @ git+https://github.com/DRuanli/dvai"
import nltk
nltk.download("cmudict", quiet=True); nltk.download("subjectivity", quiet=True); nltk.download("punkt", quiet=True); nltk.download("punkt_tab", quiet=True)
SUBJ_DIR = nltk.data.find("corpora/subjectivity")
import torch, d2vi
print("d2vi", d2vi.__version__, "| GPUs:", torch.cuda.device_count())
assert d2vi.__version__ >= "2.0.0.dev4"


In [ ]:
import glob, os, shutil, json, hashlib, time
import pandas as pd
WORK, TMP = "/kaggle/working", "/tmp/raid"
os.makedirs(TMP, exist_ok=True)
for f in glob.glob("/kaggle/input/**/k05_*", recursive=True):   # resume from an earlier run
    dst = os.path.join(WORK, os.path.basename(f))
    if os.path.isfile(f) and not os.path.exists(dst):
        shutil.copy(f, dst)
print("working dir:", sorted(os.listdir(WORK)))
RAID_URL = "https://dataset.raid-bench.xyz"


In [ ]:
# Format check of RAID on the first 20 MB of the file without attacks (quick)
from d2vi.data.heldout import RAID
!curl -s -r 0-20000000 {RAID_URL}/train_none.csv -o {TMP}/head.csv
head = pd.read_csv(f"{TMP}/head.csv", nrows=3000)
print(list(head.columns))
for col in ("model", "domain", "attack", "decoding"):
    print(col, sorted(map(str, head[col].unique())))
assert set(RAID["columns"]) <= set(head.columns), "RAID columns differ: send this output back"
assert set(head.attack) == {"none"}
print(head.generation.str.len().describe().round(0).to_dict())
print("RAID format OK")


In [ ]:
# RAID subset (CPU; downloads 0.8 GB + 11.8 GB to /tmp, about 15-25 minutes)
from d2vi.data.heldout import sample_raid, load_beemo
RAID_DOCS = f"{WORK}/k05_raid_docs.parquet"
if not os.path.exists(RAID_DOCS):
    t0 = time.time()
    !wget -nv -c -P {TMP} {RAID_URL}/train_none.csv
    !wget -nv -c -P {TMP} {RAID_URL}/train.csv
    def chunks():
        for name in ("train_none.csv", "train.csv"):
            yield from pd.read_csv(f"{TMP}/{name}", usecols=list(RAID["columns"]), chunksize=200_000)
    raid = sample_raid(chunks())
    raid.to_parquet(RAID_DOCS, index=False)
    !rm -f {TMP}/train.csv {TMP}/train_none.csv
    print("sampled in", round(time.time() - t0), "s")
raid = pd.read_parquet(RAID_DOCS)
print(raid.shape); print(raid.groupby(["source", "version"]).size().unstack(fill_value=0))
print(raid.groupby("domain").a.agg(["size", "mean"]))


In [ ]:
BEEMO_FILE = f"{WORK}/k05_beemo_dataset.parquet"
if not os.path.exists(BEEMO_FILE):
    !wget -nv -O {BEEMO_FILE} https://raw.githubusercontent.com/Toloka/beemo/main/dataset.parquet
assert hashlib.sha256(open(BEEMO_FILE, "rb").read()).hexdigest() == "5df88f7941e873db3c6e2be4e4085db50fa824f7fc626685f0e317e935a188db", "Beemo file changed"
beemo = load_beemo(BEEMO_FILE)
print(beemo.groupby("version").size().to_dict())


In [ ]:
from d2vi.features.heldout import score_documents, finalize_features, operating_points
from d2vi.features.pipeline import package_versions
VERSIONS = package_versions("d2vi", "transformers", "torch", "vaderSentiment", "textstat", "nrclex", "scikit-learn")

def build(name, docs, calibration, train_mask):
    scores = score_documents(docs, SUBJ_DIR, f"{WORK}/k05_{name}_fdgpt_cache.csv")
    table = docs.merge(scores, on="doc_id", validate="one_to_one")
    table.drop(columns=["text"]).to_parquet(f"{WORK}/k05_{name}_scores.parquet", index=False)
    feats, info = finalize_features(table, calibration, train_mask(table))
    feats.to_parquet(f"{WORK}/{name}_features.parquet", index=False)
    ops = operating_points(feats, by=("source", "version"))
    info |= {"corpus": name, "versions": VERSIONS,
             "operating_points": ops.round(4).to_dict("records")}
    json.dump(info, open(f"{WORK}/{name}_summary.json", "w"), indent=2)
    print(name, "evaluation rows:", len(feats)); print(ops.round(3).to_string(index=False), flush=True)

# Beemo first (small): a problem shows up within the first half hour
build("beemo", beemo, "split", lambda t: t["version"].eq("machine"))


In [ ]:
build("raid", raid, "cross_domain", lambda t: t["source"].eq("chatgpt") & t["version"].eq("none"))
print(sorted(os.listdir(WORK)))
